## Clark CLEAN 编程练习


本 notebook 用于实现简化的 Clark CLEAN。算法说明见 $\S$ 6.3。本练习要求补全 `selectSubPSF()` 和 `clark()`：次循环调用已测试的 Högbom 实现，主循环使用完整 PSF 由总模型重新计算残图。评分重点包括停止条件、正负分量、线性边界处理和不同输入形状下的通用性。

伪代码如下：


$\textbf{input: } I^{D}, \ B, \ \gamma, \ f_{\textrm{thresh}}, \ N_{\textrm{major}}, \ N_{\textrm{minor}}, \ \mathcal{M}$

$\textbf{initialize: } M \leftarrow 0, \ R \leftarrow I^{D}, \ i \leftarrow 0, \ (B_{\textrm{sub}},\eta) \leftarrow g(B)$

$\textbf{while} \ \max_{\mathcal{M}}|R| > f_{\textrm{thresh}} \ \textbf{and} \ i < N_{\textrm{major}} \ \textbf{do:} \quad [\textrm{major cycle}]$

$\qquad (l_p,m_p) \leftarrow \underset{(l,m)\in\mathcal{M}}{\operatorname{argmax}} |R(l,m)|, \quad r_p \leftarrow R(l_p,m_p)$

$\qquad f_{\textrm{minor}} \leftarrow \max(f_{\textrm{thresh}},\eta |r_p|)$

$\qquad \Delta M \leftarrow \textrm{Hogbom}(R,B_{\textrm{sub}},\gamma,f_{\textrm{minor}},N_{\textrm{minor}},\mathcal{M}) \quad [\textrm{minor cycle}]$

$\qquad M \leftarrow M+\Delta M$

$\qquad R \leftarrow I^D-B*M \quad [\textrm{full-PSF update}]$

$\qquad i \leftarrow i +1$

$\textbf{output: } M,R$

$\eta$ 应界定子 PSF 未表示部分的最大响应比例。主循环每次返回完整 PSF，可校正次循环的截断近似；停止判断和峰值搜索必须使用当前残图，并保留峰值符号。边界更新采用实际重叠区域，不能使用会产生环绕的循环平移。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from scipy import signal

from clean_demo import gaussian_clean_beam, hogbom_clean, load_clean_data, restore_image


In [ ]:
# The minor cycle uses clean_demo.hogbom_clean, including signed peaks and clipped boundaries.


In [ ]:
# gaussian_clean_beam and restore_image are shared with the worked Högbom example.


In [ ]:
# The restored image preserves negative components; no absolute value is applied.


In [ ]:
# Use hogbom_clean(..., psf=subPsfImg) for each minor cycle.


In [ ]:
def selectSubPSF(psfImg):
    """Select out a central region of the PSF for the Hogbom minor cycle,
    and bound the largest absolute PSF response omitted from that region.

    There are a number of ways to implement this function. A general method
    would determine where the first sidelobe is, and select out a region of
    the PSF which includes those sidelobes. A simple method would be to hard-code
    the size of the sub-image. Marks will be given based on how general the
    function is.

    inputs:
    psfImg: 2-D numpy array, PSF

    outputs:
    subPsfImg: odd-sized 2-D array centred on the normalized PSF peak.
    peakRatio: float in [0, 1), maximum omitted absolute response relative to the peak.
    """

    raise NotImplementedError("请实现子 PSF 选择与峰值比计算")


In [ ]:
def clark(dirtyImg, psfImg, gain, niter, fthresh, minor_niter=100, mask=None):
    """Return (residual, model) after at most niter major cycles."""
    searchMask = np.ones_like(dirtyImg, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    if searchMask.shape != dirtyImg.shape or not np.any(searchMask):
        raise ValueError('mask must select at least one image pixel')

    peak = np.unravel_index(np.argmax(np.abs(psfImg)), psfImg.shape)
    fullPsf = psfImg / psfImg[peak]
    subPsfImg, peakRatio = selectSubPSF(fullPsf)
    model = np.zeros_like(dirtyImg)
    residual = dirtyImg.copy()

    for _ in range(niter):
        search = np.where(searchMask, np.abs(residual), -np.inf)
        position = np.unravel_index(np.argmax(search), residual.shape)
        if np.abs(residual[position]) <= fthresh:
            break

        # TODO: run one minor cycle down to max(fthresh, peakRatio * abs(residual[position])).
        # TODO: add its model to model, then set residual = dirtyImg - fullPsf * model.
        # Use signal.fftconvolve(..., mode='same') for the linear full-PSF update.
        raise NotImplementedError('请实现 Clark 主循环')

    return residual, model


***


In [ ]:
gain = 0.1
niter = 20
fthresh = 2.5
run_student_solution = False  # 完成 selectSubPSF() 和 clark() 后改为 True


In [ ]:
dirtyImg, psfImg, dataSource = load_clean_data(
    '../data/fits/deconv/KAT-7_6h60s_dec-30_10MHz_10chans_uniform_n100-dirty.fits',
    '../data/fits/deconv/KAT-7_6h60s_dec-30_10MHz_10chans_uniform_n100-psf.fits',
)
print(f'Data source: {dataSource}')
cleanBeam = gaussian_clean_beam(psfImg)


In [ ]:
if run_student_solution:
    residImg, skyModel = clark(dirtyImg, psfImg, gain, niter, fthresh)
else:
    residImg, skyModel = dirtyImg.copy(), np.zeros_like(dirtyImg)
    print('完成两个待实现函数并启用 run_student_solution 后运行 Clark CLEAN。')


In [ ]:
#plot the dirty image
fig = plt.figure(figsize=(8,8))
plt.imshow(dirtyImg, origin='lower')
plt.title('Dirty Image')
plt.colorbar()


In [ ]:
#plot the residual image
fig = plt.figure(figsize=(8,8))
plt.imshow(residImg, origin='lower')
plt.title('Residual Image')
plt.colorbar()


In [ ]:
#plot the restored image
restImg = restore_image(skyModel, residImg, cleanBeam)
fig = plt.figure(figsize=(8,8))
plt.imshow(restImg, origin='lower')
plt.title('Restored Image')
plt.colorbar()
